# 4. SISTEM REKOMENDASI (TERPISAH)

**Ekstraksi Kata Kunci dengan N-Gram dan IndoBERT untuk Rekomendasi Wisata Bogor**

---

## Alur Sistem Rekomendasi Baru

Sistem rekomendasi dibagi menjadi dua fungsi terpisah sesuai kebutuhan aplikasi:

1.  **Pencarian (Search):** Menggunakan **IndoBERT** untuk menangkap makna semantik dari query pencarian.
2.  **Rekomendasi Detail (Related Items):** Menggunakan **N-Gram + TF-IDF** untuk mencari wisata serupa berdasarkan kemiripan kata kunci/teks pada halaman detail.

```
                    ┌──────────────────────┐
Input Query ──────► │    INDOBERT (100%)   │ ──► HASIL PENCARIAN
  (Search)          │ (Semantic Matching)  │
                    └──────────────────────┘

                    ┌──────────────────────┐
Detail Wisata ────► │ N-GRAM + TF-IDF (100%) │ ──► REKOMENDASI
 (Related)          │  (Keyword Matching)  │     (Halaman Detail)
                    └──────────────────────┘
```

In [1]:
import pandas as pd
import numpy as np
import pickle
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
import re
warnings.filterwarnings('ignore')

DATA_PATH = './data/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Libraries imported!")

c:\laragon\bin\python\python-3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported!


In [2]:
# Load semua data yang dibutuhkan
print("🔄 Loading data...\n")

df = pd.read_csv(f'{DATA_PATH}data_with_keywords.csv')
tfidf_matrix = np.load(f'{DATA_PATH}tfidf_matrix.npy')
with open(f'{DATA_PATH}tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
indobert_embeddings = np.load(f'{DATA_PATH}indobert_embeddings.npy')

# Tampilkan dalam tabel
info_df = pd.DataFrame({
    'Komponen': ['Total Wisata', 'TF-IDF Matrix', 'IndoBERT Embeddings', 'Vocabulary Size'],
    'Nilai': [len(df), str(tfidf_matrix.shape), str(indobert_embeddings.shape), len(tfidf_vectorizer.vocabulary_)]
})
print("TABEL: DATA YANG DIMUAT")
display(info_df)

🔄 Loading data...

TABEL: DATA YANG DIMUAT


,Komponen,Nilai
0,Total Wisata,296
1,TF-IDF Matrix,"(296, 5000)"
2,IndoBERT Embeddings,"(296, 768)"
3,Vocabulary Size,5000


## 4.1 Hitung Similarity Matrix (Terpisah)

In [3]:
# Hitung similarity matrix untuk kedua metode secara terpisah
print("🔄 Computing similarity matrices...\n")

# PATH 1: N-gram + TF-IDF similarity (Untuk Rekomendasi/Related Items)
ngram_similarity = cosine_similarity(tfidf_matrix)

# PATH 2: IndoBERT similarity (Untuk Pencarian/Search)
indobert_similarity = cosine_similarity(indobert_embeddings)

# Note: Kita tidak lagi menggabungkan keduanya menjadi 'combined_similarity'
# Simpan matriks terpisah ini jika diperlukan oleh API
np.save(f'{DATA_PATH}ngram_similarity.npy', ngram_similarity)
np.save(f'{DATA_PATH}indobert_similarity.npy', indobert_similarity)

# Tampilkan info
sim_info = pd.DataFrame({
    'Metode': ['N-gram + TF-IDF', 'IndoBERT'],
    'Tujuan': ['Rekomendasi (Detail Page)', 'Pencarian (Search Bar)'],
    'Tipe': ['Lexical (Kata Kunci)', 'Semantic (Makna)'],
    'Matrix Shape': [str(ngram_similarity.shape), str(indobert_similarity.shape)]
})
print("TABEL: METODE TERPISAH")
display(sim_info)

🔄 Computing similarity matrices...

TABEL: METODE TERPISAH


,Metode,Tujuan,Tipe,Matrix Shape
0,N-gram + TF-IDF,Rekomendasi (Detail Page),Lexical (Kata Kunci),"(296, 296)"
1,IndoBERT,Pencarian (Search Bar),Semantic (Makna),"(296, 296)"


## 4.2 Load IndoBERT Model

In [4]:
# Load IndoBERT untuk encode user input saat SEARCH
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def get_embedding(text):
    encoded = tokenizer(str(text), padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model(**encoded)
        embedding = mean_pooling(output, encoded['attention_mask'])
    return embedding.cpu().numpy()

print("✅ IndoBERT encoder ready!")

✅ IndoBERT encoder ready!


## 4.3 Fungsi Pencarian & Rekomendasi (Terpisah)

In [5]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def search_places(query, top_n=5):
    """
    FUNGSI 1: PENCARIAN (SEARCH)
    Menggunakan IndoBERT Embedding untuk mencari wisata berdasarkan makna query.
    """
    query_clean = preprocess_text(query)
    
    # Encode query -> IndoBERT Embedding
    query_embedding = get_embedding(query_clean)
    
    # Hitung cosine similarity dengan semua place embeddings
    # indobert_embeddings shape: (296, 768)
    # query_embedding shape: (1, 768) 
    bert_scores = cosine_similarity(query_embedding.reshape(1, -1), indobert_embeddings)[0]
    
    # Get top results
    top_indices = np.argsort(bert_scores)[::-1][:top_n]
    
    print(f"\n🔍 SEARCH QUERY (IndoBERT): {query}")
    print("="*100)
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        row = df.iloc[idx]
        results.append({
            'Rank': rank,
            'Nama Wisata': row['nama'],
            'Kategori': row['kategori'],
            'Score': round(bert_scores[idx], 4)
        })
    
    result_df = pd.DataFrame(results)
    display(result_df)
    return result_df

def get_detail_recommendations(place_name, top_n=5):
    """
    FUNGSI 2: REKOMENDASI DETAIL PAGE
    Menggunakan N-gram Similarity untuk mencari wisata serupa (keyword-based).
    """
    # Cari index tempat berdasarkan nama
    matches = df[df['nama'].str.lower() == place_name.lower()]
    if len(matches) == 0:
        print(f"❌ Tempat '{place_name}' tidak ditemukan.")
        return
    
    place_idx = matches.index[0]
    
    # Ambil skor similarity dari matriks N-gram
    # ngram_similarity shape: (296, 296)
    sim_scores = ngram_similarity[place_idx]
    
    # Sort (exclude diri sendiri)
    top_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    
    print(f"\n📄 REKOMENDASI UNTUK (N-gram): {place_name}")
    print("="*100)
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        row = df.iloc[idx]
        results.append({
            'Rank': rank,
            'Nama Wisata': row['nama'],
            'Kategori': row['kategori'],
            'Score': round(sim_scores[idx], 4)
        })
    
    result_df = pd.DataFrame(results)
    display(result_df)
    return result_df

## 4.4 Pengujian

In [ ]:
# TEST 1: Pencarian (pencarian teks bebas)
search_places("wisata air terjun yang sejuk", top_n=5)

# TEST 2: Rekomendasi (berdasarkan item)
# Ambil satu nama wisata sebagai contoh
sample_place = df['nama'].iloc[10]
get_detail_recommendations(sample_place, top_n=5)


🔍 SEARCH QUERY (IndoBERT): wisata air terjun yang sejuk


,Rank,Nama Wisata,Kategori,Score
0,1,Curug Bidadari,Alam,0.3717
1,2,Lembah Tepus,Arena,0.3716
2,3,Curug Pasir Reungit Endah,Arena,0.3716
3,4,Telaga Melimping,Olahraga,0.3715
4,5,Lembah Cisadane,Rekreasi,0.3713



📄 REKOMENDASI UNTUK (N-gram): Puncak Lalana


,Rank,Nama Wisata,Kategori,Score
0,1,Puncak Palasari,Arena,0.3365
1,2,Gunung Kencana,Olahraga,0.2602
2,3,Gunung Munara,Alam,0.2329
3,4,Bukit Alas Bandawasa,Arena,0.2165
4,5,Gunung Kapur Ciampea,Alam,0.2106


,Rank,Nama Wisata,Kategori,Score
0,1,Puncak Palasari,Arena,0.3365
1,2,Gunung Kencana,Olahraga,0.2602
2,3,Gunung Munara,Alam,0.2329
3,4,Bukit Alas Bandawasa,Arena,0.2165
4,5,Gunung Kapur Ciampea,Alam,0.2106


: 